# Interactive Data Labeling Tool (Binary Classification)

Use this notebook to manually label posts as Reliable (1) or Unreliable (0) per the annotation rubric.

**Instructions:**
1. Run Cell 1 to load your unlabeled data.
2. Run Cell 2 to review label definitions and the rubric.
3. Run Cell 3 to start the interactive labeling session.
4. For each post, apply the **Strict Binary Classification** rubric to assign 1 (Reliable) or 0 (Unreliable).
5. Run Cell 4 to save your progress.
6. Run Cell 5 to merge with any previously labeled data.


## 1. Load Raw Data

In [7]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

# Detect environment and find CSV file.
# Priority: most recent labeled batch (resume) -> multi-source training CSV -> x fetch CSV.
search_roots = [
    Path('./ml/data'),
    Path('../data'),
    Path('../../ml/data'),
]

resume_candidates = []
for root in search_roots:
    if root.exists():
        resume_candidates.extend(root.glob('x_labeled_batch_*.csv'))

resume_candidates = sorted(
    resume_candidates,
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

possible_paths = [
    str(resume_candidates[0]) if resume_candidates else None,
    './ml/data/x_multi_source_training.csv',
    '../data/x_multi_source_training.csv',
    '../../ml/data/x_multi_source_training.csv',
    './ml/outputs/x_fetch/x_recent_posts_translated.csv',
    '/content/drive/MyDrive/x_multi_source_training.csv',  # Colab
]

CSV_PATH = None
for path in possible_paths:
    if path and Path(path).exists():
        CSV_PATH = Path(path)
        print(f'Found data at: {CSV_PATH}')
        break

if CSV_PATH is None:
    print('Data file not found in expected locations.')
    print('Please provide the full path to your CSV file.')
    csv_input = input('Path: ').strip()
    CSV_PATH = Path(csv_input)
    if not CSV_PATH.exists():
        raise FileNotFoundError(f'File not found: {csv_input}')

df = pd.read_csv(CSV_PATH)

# Normalize common schema differences
if 'x_post_id' in df.columns and 'post_id' not in df.columns:
    df = df.rename(columns={'x_post_id': 'post_id'})

for col in ['post_id', 'created_at', 'raw_text']:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if 'translated_text' not in df.columns:
    df['translated_text'] = df['raw_text']

if 'source_type' not in df.columns:
    df['source_type'] = 'social_media'

if 'source_file' not in df.columns:
    df['source_file'] = ''

if 'reliability_label' not in df.columns:
    df['reliability_label'] = pd.NA
if 'annotator_name' not in df.columns:
    df['annotator_name'] = ''
if 'annotation_time' not in df.columns:
    df['annotation_time'] = pd.NA
if 'notes' not in df.columns:
    df['notes'] = ''

print(f'Loaded {len(df)} posts')
print(f'Columns: {list(df.columns)}')
print('Source distribution:')
print(df['source_type'].value_counts(dropna=False))

df.head()

Found data at: ..\data\x_labeled_batch_20260511_1852_auto_bootstrap.csv
Loaded 210 posts
Columns: ['post_id', 'created_at', 'lang', 'raw_text', 'translated_text', 'source_type', 'source_file', 'reliability_label', 'annotator_name', 'annotation_time', 'notes']
Source distribution:
source_type
social_media    198
gdelt            12
Name: count, dtype: int64


,post_id,created_at,lang,raw_text,translated_text,source_type,source_file,reliability_label,annotator_name,annotation_time,notes
0,gdelt_0,20260511T050000Z,en,"[GDELT] In new signature drive , Filipino Chur...","[GDELT] In new signature drive , Filipino Chur...",gdelt,NaN,0.0,auto_bootstrap,2026-05-11T18:52:18,auto: gdelt non-traffic context
1,gdelt_1,20260510T101500Z,en,[GDELT] VinFast VF6 proves EVs go further deli...,[GDELT] VinFast VF6 proves EVs go further deli...,gdelt,NaN,NaN,NaN,NaN,NaN
2,gdelt_2,20260510T073000Z,en,[GDELT] [ Time Trowel ] What happens when we r...,[GDELT] [ Time Trowel ] What happens when we r...,gdelt,NaN,0.0,auto_bootstrap,2026-05-11T18:52:18,auto: gdelt non-traffic context
3,gdelt_3,20260509T213000Z,en,"[GDELT] Successful ASEAN Summit , a wow moment...","[GDELT] Successful ASEAN Summit , a wow moment...",gdelt,NaN,0.0,auto_bootstrap,2026-05-11T18:52:18,auto: gdelt non-traffic context
4,gdelt_4,20260509T063000Z,en,[GDELT] Quezon City roads closed Sunday for ru...,[GDELT] Quezon City roads closed Sunday for ru...,gdelt,NaN,1.0,auto_bootstrap,2026-05-11T18:52:18,auto: gdelt traffic/event + PH location


## 2. Interactive Labeling Session

In [8]:
# Prepare labeling dataframe (resume-safe for existing labels)
label_df = df[[
    'post_id',
    'created_at',
    'raw_text',
    'translated_text',
    'source_type',
    'source_file',
    'reliability_label',
    'annotator_name',
    'annotation_time',
    'notes',
]].copy()

label_df['reliability_label'] = pd.to_numeric(label_df['reliability_label'], errors='coerce')
label_df['annotator_name'] = label_df['annotator_name'].fillna('').astype(str)
label_df['notes'] = label_df['notes'].fillna('').astype(str)

# Annotation controls
ANNOTATION_LIMIT = 80  # set to None to label all pending rows
PREFER_RECENT_FIRST = True

if PREFER_RECENT_FIRST:
    label_df = label_df.sort_values('created_at', ascending=False).reset_index(drop=True)

pending_mask = label_df['reliability_label'].isna()
pending_total = int(pending_mask.sum())

if ANNOTATION_LIMIT is not None:
    pending_indices = label_df[pending_mask].head(ANNOTATION_LIMIT).index.tolist()
else:
    pending_indices = label_df[pending_mask].index.tolist()

print(f'Total rows: {len(label_df)}')
print(f'Already labeled: {len(label_df) - pending_total}')
print(f'Pending labels: {pending_total}')
print(f'Current annotation queue: {len(pending_indices)} posts')
print()
print('Queue source breakdown:')
print(label_df.loc[pending_indices, 'source_type'].value_counts(dropna=False))
print()
print('=' * 80)
print('STRICT BINARY CLASSIFICATION RUBRIC')
print('=' * 80)
print()
print('reliability_label: 1 = Reliable | 0 = Unreliable')
print()
print('1 = RELIABLE:')
print('   The text contains trustworthy, factual, and actionable information')
print('   regarding a traffic-disrupting event with specific geographic details.')
print()
print('0 = UNRELIABLE:')
print('   The text is misinformation, subjective noise, irrelevant, or non-specific.')
print()
print('ANNOTATION RULES BY SOURCE:')
print('  - Institutional sources (MMDA, PNP, PAGASA, News API): provisional 1 if event is specific')
print('  - GDELT data: evaluate relevance and local context; do not auto-score as reliable')
print('  - Commuter posts: 1 only if objective report with location + incident type')
print('  - Taglish/informal: evaluate semantics, not style')
print()
print('=' * 80)
print()
print('Run the next cell to begin interactive labeling.')

Total rows: 210
Already labeled: 163
Pending labels: 47
Current annotation queue: 47 posts

Queue source breakdown:
source_type
social_media    40
gdelt            7
Name: count, dtype: int64

STRICT BINARY CLASSIFICATION RUBRIC

reliability_label: 1 = Reliable | 0 = Unreliable

1 = RELIABLE:
   The text contains trustworthy, factual, and actionable information
   regarding a traffic-disrupting event with specific geographic details.

0 = UNRELIABLE:
   The text is misinformation, subjective noise, irrelevant, or non-specific.

ANNOTATION RULES BY SOURCE:
  - Institutional sources (MMDA, PNP, PAGASA, News API): provisional 1 if event is specific
  - GDELT data: evaluate relevance and local context; do not auto-score as reliable
  - Commuter posts: 1 only if objective report with location + incident type
  - Taglish/informal: evaluate semantics, not style


Run the next cell to begin interactive labeling.


In [12]:
# Interactive labeling loop - Strict Binary Classification
annotator = input('Enter your name for annotation: ').strip()
if not annotator:
    annotator = 'annotator'

labeled_count = 0
skipped_count = 0

for i, idx in enumerate(pending_indices, start=1):
    row = label_df.loc[idx]

    print(f'\n--- POST {i}/{len(pending_indices)} ---')
    print(f"ID: {row['post_id']}")
    print(f"Time: {row['created_at']}")
    print(f"Source: {row['source_type']}")

    if str(row.get('source_file', '')).strip():
        print(f"Source file: {row['source_file']}")

    print()
    print('RAW TEXT:')
    raw_text = str(row['raw_text'])
    raw_display = raw_text if raw_text and raw_text.lower() != 'nan' else '[missing raw text]'
    print(f"  {raw_display[:350]}..." if len(raw_display) > 350 else f"  {raw_display}")

    print()
    print('TRANSLATED TEXT:')
    translated_value = row.get('translated_text', '')
    translated_text = '' if pd.isna(translated_value) else str(translated_value)
    translated_display = translated_text if translated_text and translated_text.lower() != 'nan' else raw_display
    print(f"  {translated_display[:350]}..." if len(translated_display) > 350 else f"  {translated_display}")
    print()

    while True:
        rel_input = input('Reliable? (1=yes, 0=no, skip=skip, quit=save and stop): ').strip().lower()

        if rel_input == 'skip':
            skipped_count += 1
            break

        if rel_input == 'quit':
            print('Stopping early. You can save in the next cell.')
            i = len(pending_indices)
            break

        if rel_input in {'0', '1'}:
            reliability_label = int(rel_input)
            notes = input('Notes (optional): ').strip()

            label_df.loc[idx, 'reliability_label'] = reliability_label
            label_df.loc[idx, 'annotator_name'] = annotator
            label_df.loc[idx, 'annotation_time'] = pd.Timestamp.now().isoformat(timespec='seconds')
            label_df.loc[idx, 'notes'] = notes
            labeled_count += 1
            break

        print('  Invalid. Enter 1, 0, skip, or quit.')

    if rel_input == 'quit':
        break

    if (labeled_count + skipped_count) % 10 == 0:
        print(f"  [Progress: {labeled_count} labeled, {skipped_count} skipped]")

print('\n=== LABELING SESSION COMPLETE ===')
print(f'Total newly labeled this session: {labeled_count}')
print(f'Total skipped this session: {skipped_count}')
print(f'Total labeled in dataframe: {int(label_df["reliability_label"].notna().sum())}/{len(label_df)}')
print(f'Reliable (1): {int((label_df["reliability_label"] == 1).sum())}')
print(f'Unreliable (0): {int((label_df["reliability_label"] == 0).sum())}')


--- POST 1/47 ---
ID: gdelt_1
Time: 20260510T101500Z
Source: gdelt
Source file: nan

RAW TEXT:
  [GDELT] VinFast VF6 proves EVs go further delivers real - world performance beyond city driving (Source: https://gadgetsmagazine.com.ph/mobility/cars/vinfast-vf6-evs-go-further)

TRANSLATED TEXT:
  [GDELT] VinFast VF6 proves EVs go further delivers real - world performance beyond city driving (Source: https://gadgetsmagazine.com.ph/mobility/cars/vinfast-vf6-evs-go-further)


--- POST 2/47 ---
ID: gdelt_5
Time: 20260508T203000Z
Source: gdelt
Source file: nan

RAW TEXT:
  [GDELT] Fireworks memorial returns to honour Tyendinaga fishermen and keep questions alive (Source: https://theturtleislandnews.com/index.php/2026/05/08/fireworks-memorial-returns-to-honour-tyendinaga-fishermen-and-keep-questions-alive/)

TRANSLATED TEXT:
  [GDELT] Fireworks memorial returns to honour Tyendinaga fishermen and keep questions alive (Source: https://theturtleislandnews.com/index.php/2026/05/08/fireworks-memori

## 3. Save Labeled Data

In [13]:
# Save this batch
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
batch_filename = f'x_labeled_batch_{timestamp}_{annotator}.csv'

# Determine save path (local first, fallback to Colab)
possible_save_dirs = [
    Path('./ml/data'),
    Path('../data'),
    Path('../../ml/data'),
    Path('/content/drive/MyDrive'),  # Colab
]

save_dir = None
for d in possible_save_dirs:
    try:
        d.mkdir(parents=True, exist_ok=True)
        save_dir = d
        break
    except:
        continue

if save_dir is None:
    save_dir = Path('./ml/data')
    save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / batch_filename
label_df.to_csv(save_path, index=False)
print(f'Saved to: {save_path}')

# Summary statistics
labeled = label_df[label_df['reliability_label'].notna()]
print('\nLabeled Summary:')
print(f'  Total labeled: {len(labeled)}')
print(f'  Reliable (1): {(labeled["reliability_label"] == 1).sum()}')
print(f'  Unreliable (0): {(labeled["reliability_label"] == 0).sum()}')
if len(labeled) > 0:
    print(f'  Reliability rate: {(labeled["reliability_label"] == 1).sum() / len(labeled) * 100:.1f}%')
else:
    print('  Reliability rate: N/A (no labels yet)')

Saved to: ml\data\x_labeled_batch_20260511_1934_Joshua Pagallaman.csv

Labeled Summary:
  Total labeled: 210
  Reliable (1): 168
  Unreliable (0): 42
  Reliability rate: 80.0%


## 4. Merge with Historical Labels (Optional)

In [14]:
# Load any existing labeled master file
possible_master_paths = [
    Path('./ml/data/x_labeled_master.csv'),
    Path('../data/x_labeled_master.csv'),
    Path('../../ml/data/x_labeled_master.csv'),
    Path('/content/drive/MyDrive/x_labeled_master.csv'),  # Colab
]

master_path = None
for p in possible_master_paths:
    if p.exists():
        master_path = p
        break

if master_path is None:
    # Default to local path
    master_path = Path('./ml/data/x_labeled_master.csv')

if master_path.exists():
    master_df = pd.read_csv(master_path)
    print(f'Loaded existing master with {len(master_df)} rows')
    
    # Merge: take labeled rows from current batch
    new_labeled = label_df[label_df['reliability_label'].notna()].copy()
    
    # Remove any duplicates in master (by post_id)
    master_df = master_df[~master_df['post_id'].isin(new_labeled['post_id'])]
    
    # Append and save
    merged = pd.concat([master_df, new_labeled], ignore_index=True)
    merged = merged.sort_values('created_at')
    
    # Ensure parent dir exists
    master_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(master_path, index=False)
    
    print(f'Updated master now has {len(merged)} total labeled rows')
    print(f'  Reliable (1): {(merged["reliability_label"] == 1).sum()}')
    print(f'  Unreliable (0): {(merged["reliability_label"] == 0).sum()}')
    print(f'  Reliability rate: {(merged["reliability_label"] == 1).sum() / len(merged) * 100:.1f}%')
else:
    # Create new master from current batch
    new_labeled = label_df[label_df['reliability_label'].notna()].copy()
    
    # Ensure parent dir exists
    master_path.parent.mkdir(parents=True, exist_ok=True)
    new_labeled.to_csv(master_path, index=False)
    print(f'Created new master with {len(new_labeled)} labeled rows')
    print(f'  Reliable (1): {(new_labeled["reliability_label"] == 1).sum()}')
    print(f'  Unreliable (0): {(new_labeled["reliability_label"] == 0).sum()}')
    print(f'  Reliability rate: {(new_labeled["reliability_label"] == 1).sum() / len(new_labeled) * 100:.1f}%')

print(f'\nMaster saved to: {master_path}')


Created new master with 210 labeled rows
  Reliable (1): 168
  Unreliable (0): 42
  Reliability rate: 80.0%

Master saved to: ml\data\x_labeled_master.csv


## 5. Inspect Master Dataset

In [15]:
# Show summary of master labeled data
possible_master_paths = [
    Path('./ml/data/x_labeled_master.csv'),
    Path('../data/x_labeled_master.csv'),
    Path('../../ml/data/x_labeled_master.csv'),
    Path('/content/drive/MyDrive/x_labeled_master.csv'),  # Colab
]

master_path = None
for p in possible_master_paths:
    if p.exists():
        master_path = p
        break

if master_path and master_path.exists():
    master = pd.read_csv(master_path)
    print(f'Master dataset: {len(master)} labeled posts')
    print()
    print('Reliability distribution:')
    dist = master['reliability_label'].value_counts().sort_index()
    print(dist)
    print()
    reliable_count = (master['reliability_label'] == 1).sum()
    total_count = len(master)
    print(f'Reliability rate: {reliable_count}/{total_count} = {reliable_count/total_count*100:.1f}%')
    print()
    print('Sample of labeled data:')
    display(master[['post_id', 'created_at', 'reliability_label', 'annotator_name', 'notes']].head(10))
else:
    print('Master dataset not found yet. Run the labeling cells first.')


Master dataset: 210 labeled posts

Reliability distribution:
reliability_label
0.0     42
1.0    168
Name: count, dtype: int64

Reliability rate: 168/210 = 80.0%

Sample of labeled data:


,post_id,created_at,reliability_label,annotator_name,notes
0,gdelt_0,20260511T050000Z,0.0,auto_bootstrap,auto: gdelt non-traffic context
1,gdelt_1,20260510T101500Z,0.0,Joshua Pagallaman,Advertisement Post
2,gdelt_2,20260510T073000Z,0.0,auto_bootstrap,auto: gdelt non-traffic context
3,gdelt_3,20260509T213000Z,0.0,auto_bootstrap,auto: gdelt non-traffic context
4,gdelt_4,20260509T063000Z,1.0,auto_bootstrap,auto: gdelt traffic/event + PH location
5,gdelt_5,20260508T203000Z,0.0,Joshua Pagallaman,Not related to PH
6,gdelt_6,20260508T194500Z,0.0,Joshua Pagallaman,NaN
7,gdelt_7,20260508T183000Z,0.0,Joshua Pagallaman,Not related
8,gdelt_8,20260508T164500Z,0.0,auto_bootstrap,auto: gdelt non-traffic context
9,gdelt_9,20260508T154500Z,0.0,Joshua Pagallaman,Not in PH
